In [41]:
import pandas as pd
import numpy as np

# loading csv file
df = pd.read_csv("digital_payment_transactions.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 303000 entries, 0 to 302999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   transaction_id     303000 non-null  int64  
 1   user_id            303000 non-null  int64  
 2   transaction_date   303000 non-null  str    
 3   amount             303000 non-null  float64
 4   transaction_type   303000 non-null  str    
 5   city               299966 non-null  str    
 6   device_type        299969 non-null  str    
 7   merchant_category  303000 non-null  str    
 8   is_fraud           303000 non-null  int64  
dtypes: float64(1), int64(3), str(5)
memory usage: 20.8 MB


In [42]:
df.head()

,transaction_id,user_id,transaction_date,amount,transaction_type,city,device_type,merchant_category,is_fraud
0,1,16795,2023-03-02 02:37:00,39.71,UPI,Bangalore,Web,Electronics,0
1,2,1860,2023-11-15 04:59:00,263.44,UPI,Hyderabad,Android,Shopping,0
2,3,6390,2023-12-22 23:12:00,2581.11,Card,Ahmedabad,Android,Bills,0
3,4,12964,2023-06-11 00:39:00,591.72,Wallet,Ahmedabad,Android,Food,0
4,5,12284,2023-09-19 23:01:00,2905.57,UPI,Mumbai,Android,Entertainment,0


In [43]:
df.tail()

,transaction_id,user_id,transaction_date,amount,transaction_type,city,device_type,merchant_category,is_fraud
302995,116691,8636,2023-02-18 03:41:00,1034.97,Card,Pune,NaN,Grocery,0
302996,257875,12536,2023-12-19 03:13:00,3213.71,Card,Mumbai,Android,Shopping,0
302997,222866,14003,2023-07-02 05:40:00,1682.35,UPI,Hyderabad,Web,Entertainment,0
302998,107111,7000,2023-09-26 01:57:00,1391.78,UPI,Hyderabad,Android,Food,0
302999,138190,7665,2023-01-08 02:01:00,510.87,Card,Pune,iOS,Food,0


# Data inspection

## Check Missing Values (% per column)

In [44]:
# Count null values
null_counts = df.isnull().sum()

# Null percentage
null_percentage = (df.isnull().sum() / len(df)) * 100

null_report = pd.DataFrame({
    "Null_Count": null_counts,
    "Null_Percentage": null_percentage
})

null_report

,Null_Count,Null_Percentage
transaction_id,0,0.00000
user_id,0,0.00000
transaction_date,0,0.00000
amount,0,0.00000
transaction_type,0,0.00000
city,3034,1.00132
device_type,3031,1.00033
merchant_category,0,0.00000
is_fraud,0,0.00000


## Check Duplicate rows 

In [45]:
print(f"Total rows before removing duplicate: {len(df)}")

# Check duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate Rows:", duplicate_count)

Total rows before removing duplicate: 303000
Duplicate Rows: 3000


In [46]:
# Remove duplicates
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 300000


## Validate & Correct Data Types

In [47]:
# Convert transaction_date to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

# Ensure amount is numeric
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

# Convert is_fraud to integer
df['is_fraud'] = df['is_fraud'].astype(int)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   transaction_id     300000 non-null  int64         
 1   user_id            300000 non-null  int64         
 2   transaction_date   300000 non-null  datetime64[us]
 3   amount             300000 non-null  float64       
 4   transaction_type   300000 non-null  str           
 5   city               297000 non-null  str           
 6   device_type        297000 non-null  str           
 7   merchant_category  300000 non-null  str           
 8   is_fraud           300000 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(4)
memory usage: 20.6 MB


## Filling missing values

In [48]:
df['city'] = df['city'].fillna('Unknown')
df['device_type'] = df['device_type'].fillna('Unknown')

# Re-check nulls
df.isnull().sum()

transaction_id       0
user_id              0
transaction_date     0
amount               0
transaction_type     0
city                 0
device_type          0
merchant_category    0
is_fraud             0
dtype: int64

# Basic EDA


### What type of transactions are customers doing most?


In [49]:
transaction_type_percent = df['transaction_type'].value_counts(normalize=True) * 100
transaction_type_percent

transaction_type
UPI           50.004667
Card          25.026333
Wallet        14.992667
NetBanking     9.976333
Name: proportion, dtype: float64

### What is overall fraud ratio?

In [50]:
fraud_counts = df['is_fraud'].value_counts()
fraud_per=df['is_fraud'].value_counts(normalize=True)*100

print(f"Fraud count= {fraud_counts[1]} \nFraud percentage= {fraud_per[1]}")


Fraud count= 2265 
Fraud percentage= 0.755


### Cities by Transaction Volume

In [51]:
df['city'].value_counts()

city
Chennai      37261
Bangalore    37225
Mumbai       37224
Delhi        37222
Pune         37131
Kolkata      37129
Hyderabad    36973
Ahmedabad    36835
Unknown       3000
Name: count, dtype: int64

In [52]:
df['city'].value_counts(normalize=True)*100

city
Chennai      12.420333
Bangalore    12.408333
Mumbai       12.408000
Delhi        12.407333
Pune         12.377000
Kolkata      12.376333
Hyderabad    12.324333
Ahmedabad    12.278333
Unknown       1.000000
Name: proportion, dtype: float64

### Fraud Rate by City

In [53]:
fraud_by_city = df.groupby('city')['is_fraud'].mean() *100
fraud_by_city.sort_values(ascending=False)


city
Chennai      0.866858
Delhi        0.840901
Bangalore    0.741437
Pune         0.732542
Kolkata      0.727194
Mumbai       0.719966
Ahmedabad    0.716710
Unknown      0.700000
Hyderabad    0.697807
Name: is_fraud, dtype: float64

In [54]:
fraud_count_by_city = df[df['is_fraud'] == 1].groupby('city').size()
print(fraud_count_by_city)

city
Ahmedabad    264
Bangalore    276
Chennai      323
Delhi        313
Hyderabad    258
Kolkata      270
Mumbai       268
Pune         272
Unknown       21
dtype: int64


### Fraud by Device Type

In [55]:
fraud_by_device = df.groupby('device_type')['is_fraud'].mean() * 100
fraud_by_device

device_type
Android    0.446079
Unknown    0.533333
Web        2.485228
iOS        0.473037
Name: is_fraud, dtype: float64

In [56]:
df[df['is_fraud']==1].groupby('device_type').size()

device_type
Android     797
Unknown      16
Web        1102
iOS         350
dtype: int64

In [57]:
df['amount'].describe()

count    300000.000000
mean       2004.643807
std        2008.468379
min           0.000000
25%         577.607500
50%        1386.310000
75%        2776.762500
max       24384.270000
Name: amount, dtype: float64

In [58]:
df.to_csv("digital_payment_transactions_clean.csv" , index=False)